# AIPerf Use Case 1 — Simple Synthetic Profiling (+ Concurrency Sweep / Pareto)

Profile an OpenAI-compatible endpoint with fully **synthetic traffic** (fixed input/output sequence lengths), then extend the same command into a **concurrency sweep** to draw the cost-vs-UX Pareto curve.

These use-case notebooks demonstrate AIPerf capabilities directly.

## Table of contents

1. [Overview](#1-overview)
2. [Requirements](#2-requirements)
3. [Input — synthetic prompts](#3-input--synthetic-prompts)
4. [Test run](#4-test-run)
5. [Results analysis](#5-results-analysis)
6. [Concurrency sweep and Pareto curve](#6-concurrency-sweep-and-pareto-curve)

## 1. Overview

`aiperf profile` simulates N concurrent users (`--concurrency`) sending a total of `--request-count` requests, each with a synthetic prompt of `--synthetic-input-tokens-mean` tokens (ISL) and a forced generation length of `--output-tokens-mean` tokens (OSL). With `--streaming` enabled, AIPerf measures:

- **TTFT** (time to first token) — responsiveness before generation starts.
- **ITL** (inter-token latency) — streaming smoothness.
- **Request latency**, **output token throughput** (system-wide), **request throughput**.

Two details that are easy to get wrong:

- `--streaming` is what makes TTFT/ITL measurable at all.
- `--tokenizer` must match the served model, or token counts (and therefore every tokens/sec number) are wrong.

Reasoning models: AIPerf separates reasoning tokens from visible output tokens by default; thinking tokens count toward OSL.

## 2. Requirements

- `aiperf` CLI — the setup cell below installs it via pip if it isn't already on `PATH`. Pin whatever version you use (repo convention: record the AIPerf version per run) — this notebook was written against `release/0.3.0`.
- A reachable OpenAI-compatible endpoint (NIM, vLLM, TGI, ...) serving the model under test.
- Notebook Python deps: `pip install -r notebooks/requirements.txt` (jupyter, pandas, matplotlib).
- A Hugging Face token (`HF_TOKEN` in the config cell) if the tokenizer repo is gated (e.g. Llama, Mistral).
- Tip: AIPerf's default `--ui-type dashboard` is a live Textual TUI built for a real terminal — it can render as garbled/duplicated lines inside a Jupyter output cell. AIPerf auto-detects a non-TTY subprocess and should already fall back on its own, but if it doesn't, force it explicitly with `--ui-type none` (plain log lines) or `--ui-type simple` (tqdm-style progress bars).


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import urllib.request
from pathlib import Path

# Notebook is expected to run from notebooks/ inside the repo (Jupyter's default cwd).
REPO_ROOT = Path.cwd().parent if not (Path.cwd() / "model-selection").exists() else Path.cwd()
assert (REPO_ROOT / "model-selection").exists(), (
    f"Could not find the repo root from {Path.cwd()} — run this notebook from the notebooks/ directory."
)
print(f"Repo root: {REPO_ROOT}")

if shutil.which("aiperf") is None:
    print("aiperf CLI not found — installing into this kernel's environment ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "aiperf"], check=True)

aiperf_path = shutil.which("aiperf")
assert aiperf_path, "aiperf still not found after install — restart the kernel and rerun this cell."
print(f"aiperf found at: {aiperf_path}")
version = subprocess.run(["aiperf", "--version"], capture_output=True, text=True)
print((version.stdout or version.stderr).strip())


## 3. Input — synthetic prompts

There is no prompt file in this use case: AIPerf generates random token sequences of the requested length. That makes runs perfectly repeatable and comparable, at the cost of realism — no natural language, no shared prefixes, so KV-cache/prefix-reuse effects are invisible (see the Use Case 3 notebook for trace replay, which fixes that).

| Flag | Meaning |
|---|---|
| `--synthetic-input-tokens-mean` | ISL — prompt length in tokens (sometimes abbreviated `--isl`) |
| `--synthetic-input-tokens-stddev` | Sample ISL from a distribution instead of a fixed value |
| `--output-tokens-mean` | OSL — forced generation length (sometimes abbreviated `--osl`) |
| `--output-tokens-stddev` | Sample OSL from a distribution |
| `--random-seed` | Make the synthetic generation reproducible |

Example: ISL 1000 / OSL 500 ≈ "600 words in, 300 words back", i.e. a plausible chat-style exchange.

## 4. Test run

Set `URL`, then run — the model name is read automatically from the endpoint's `/v1/models` listing (set `MODEL` manually only to override, e.g. when the server hosts several models). Defaults below are modest (concurrency 10 / 100 requests) so the first run finishes quickly against most endpoints — scale them up as your deployment allows.

In [ ]:
# ---- Required ----------------------------------------------------------
URL = ""         # e.g. "http://localhost:8000"
MODEL = ""       # leave empty to auto-detect from {URL}/v1/models; set to override

# ---- Workload ----------------------------------------------------------
CONCURRENCY = 10          # number of simulated concurrent users
REQUEST_COUNT = 100       # total requests to send across all users
ISL = 1000                # input sequence length (tokens)
OSL = 500                 # output sequence length (tokens)
TOKENIZER = ""            # HF repo id or local dir; empty = use MODEL as tokenizer name
RANDOM_SEED = "42"

# ---- Hugging Face ------------------------------------------------------
# aiperf downloads the tokenizer from Hugging Face. A token is only needed
# for gated repos (e.g. Llama, Mistral) or private tokenizers.
HF_TOKEN = ""
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN  # inherited by the aiperf subprocess

# Relative to REPO_ROOT (aiperf runs with cwd=REPO_ROOT) — keep artifacts under notebooks/.
OUTPUT_DIR = "notebooks/artifacts/aiperf-uc1-synthetic"


In [ ]:
assert URL, "Set URL in the config cell above."

# Ask the endpoint what it serves — verifies the URL is reachable and spares
# the user a second copy-paste. Multi-model servers: first model wins.
if not MODEL:
    models_url = URL.rstrip("/") + "/v1/models"
    with urllib.request.urlopen(models_url, timeout=10) as resp:
        served = json.load(resp)["data"]
    assert served, f"{models_url} is reachable but lists no models."
    if len(served) > 1:
        print(f"{models_url} lists {len(served)} models: {[m['id'] for m in served]}")
    MODEL = served[0]["id"]
print(f"URL   : {URL}")
print(f"MODEL : {MODEL}")

# The command, written exactly as you would type it in a terminal.
# .split() turns it into the argument list subprocess expects.
cmd = f"""
aiperf profile
    --model {MODEL}
    --url {URL}
    --endpoint-type chat
    --streaming
    --concurrency {CONCURRENCY}
    --request-count {REQUEST_COUNT}
    --synthetic-input-tokens-mean {ISL}
    --output-tokens-mean {OSL}
    --random-seed {RANDOM_SEED}
    --tokenizer {TOKENIZER or MODEL}
    --artifact-dir {OUTPUT_DIR}
""".split()

print("$ " + " ".join(cmd))
result = subprocess.run(cmd, cwd=REPO_ROOT, text=True)
print(f"\nexit code: {result.returncode}")

# NOTE: --synthetic-input-tokens-mean / --output-tokens-mean are the canonical
# long names AIPerf uses; some references shorthand them as --isl / --osl. Used
# here because they are guaranteed across AIPerf versions and match this repo's
# scenario scripts (e.g. --output-tokens-mean in model-selection/scripts/run_content_generation.sh).


## 5. Results analysis

AIPerf writes three exports to the artifact directory:

- `profile_export_aiperf.csv` — aggregated statistics.
- `profile_export_aiperf.json` — the same statistics plus the run's full configuration.
- `profile_export.jsonl` — one record **per request** (analyzed in depth in the Use Case 2 notebook).

The CSV is not a single table — it contains up to three **blank-line-separated sections**:

1. **Per-request statistics** — avg/min/max/percentiles/std per metric (TTFT, ITL, request latency, per-user throughput, ...).
2. **Run totals** — single-value rows (`Metric,Value`): benchmark duration, request count, **system-wide output token throughput and request throughput**.
3. **GPU telemetry** — only populated when GPU metrics collection is configured.

In [ ]:
from io import StringIO

import pandas as pd

artifact_dir = REPO_ROOT / OUTPUT_DIR
csv_path = next(artifact_dir.rglob("profile_export_aiperf.csv"), None)
assert csv_path is not None, f"No profile_export_aiperf.csv under {artifact_dir} — did the Test run complete?"


def read_export(path):
    """Split the multi-section AIPerf CSV into (stats, totals) DataFrames.

    Section 1: per-request statistics (Metric, avg, min, max, p50, ..., std).
    Section 2: single-value run totals (Metric, Value).
    Section 3 (GPU telemetry, if present) is ignored here.
    """
    sections = Path(path).read_text().strip().split("\n\n")
    stats = pd.read_csv(StringIO(sections[0]))
    totals = (pd.read_csv(StringIO(sections[1]))
              if len(sections) > 1 else pd.DataFrame(columns=["Metric", "Value"]))
    return stats, totals


stats, totals = read_export(csv_path)
pd.set_option("display.max_rows", None)

# Section 1: per-request statistics — one row per metric (TTFT, ITL, request
# latency, per-user throughput, ...), columns are avg/min/max/percentiles/std
# computed across all individual requests in the run.
print("Per-request statistics (one row per metric, columns = avg/min/max/percentiles/std):")
display(stats)

# Section 2: run totals — single-value rows (Metric, Value) describing the
# run as a whole: benchmark duration, request count, and system-wide
# (aggregated across all concurrent users) output token throughput / request throughput.
print("Run totals (single-value rows: duration, request count, system-wide throughput):")
totals


In [ ]:
# Latency metrics (per-request statistics section) — the UX side of the story.
key_metric_patterns = [
    "time to first token",
    "inter token latency",
    "request latency",
]
mask = stats["Metric"].str.lower().apply(lambda n: any(p in n for p in key_metric_patterns))
print("UX-relevant latency metrics — avg/p50/p90/p99 across all requests (lower is better):")
display(stats.loc[mask, ["Metric", "avg", "p50", "p90", "p99"]])

# Throughput (run-totals section) — the system-wide side of the story.
print("System-wide throughput and run summary (higher throughput is better):")
totals[totals["Metric"].str.lower().str.contains("throughput|request count|duration")]


## 6. Concurrency sweep and Pareto curve

Rerunning the identical benchmark at several concurrency levels shows the fundamental trade-off:

- **TPS per GPU** (system throughput ÷ GPU count) — resource/cost efficiency, improves with concurrency up to a saturation point.
- **TPS per user** (throughput ÷ concurrency) — individual user experience, degrades monotonically with concurrency.

Per-GPU efficiency typically rises with concurrency up to a saturation point and can *regress* past it (e.g. client-side network/port-exhaustion effects), while TTFT grows steadily. The curve is how you pick an operating point and set a per-replica load-balancer cap. This is the same idea as the repo's `sizing/` concurrency ladder (1/5/10/25/50/100/200) — use that suite for the real sizing workflow.

The sweep below is **off by default** (`RUN_SWEEP = False`) because it reruns the full benchmark once per level. Setting it to `True` makes the next cell loop over `SWEEP_CONCURRENCIES`, rerunning the same `aiperf profile` command from [section 4](#4-test-run) once per concurrency level (each run gets its own `{OUTPUT_DIR}-sweep-c<N>` artifact directory so results don't overwrite each other). The cell after that then reads every sweep artifact back in, computes `tps_per_gpu` / `tps_per_user` / TTFT per level, and plots the Pareto curve.

In [ ]:
RUN_SWEEP = False
SWEEP_CONCURRENCIES = [1, 5, 10, 25]   # concurrency levels to sweep
NUM_GPUS = 1                           # GPUs behind the endpoint, for the TPS/GPU axis

if RUN_SWEEP:
    for c in SWEEP_CONCURRENCIES:
        sweep_cmd = list(cmd)
        sweep_cmd[sweep_cmd.index("--concurrency") + 1] = str(c)
        sweep_cmd[sweep_cmd.index("--artifact-dir") + 1] = f"{OUTPUT_DIR}-sweep-c{c}"
        print("$ " + " ".join(sweep_cmd))
        subprocess.run(sweep_cmd, cwd=REPO_ROOT, text=True)
else:
    print("RUN_SWEEP is False — flip it to True to run the sweep.")


In [ ]:
import matplotlib.pyplot as plt

rows = []
for c in SWEEP_CONCURRENCIES:
    d = REPO_ROOT / f"{OUTPUT_DIR}-sweep-c{c}"
    p = next(d.rglob("profile_export_aiperf.csv"), None)
    if p is None:
        continue
    s, t = read_export(p)  # from the Results analysis cell
    # System-wide "Output Token Throughput (tokens/sec)" lives in the totals section.
    tps_rows = t[t["Metric"].str.lower().str.contains("output token throughput")]
    total_tps = float(tps_rows["Value"].iloc[0]) if not tps_rows.empty else None
    ttft_rows = s[s["Metric"].str.lower().str.contains("time to first token")]
    rows.append({
        "concurrency": c,
        "total_tps": total_tps,
        "tps_per_gpu": total_tps / NUM_GPUS if total_tps else None,
        "tps_per_user": total_tps / c if total_tps else None,
        "ttft_avg_ms": float(ttft_rows["avg"].iloc[0]) if not ttft_rows.empty else None,
    })

sweep = pd.DataFrame(rows)
if sweep.empty:
    print("No sweep artifacts found — run the sweep cell first.")
else:
    display(sweep)
    ax = sweep.plot(x="tps_per_user", y="tps_per_gpu", marker="o", legend=False, figsize=(7, 4.5))
    for _, r in sweep.iterrows():
        ax.annotate(f"c={int(r.concurrency)}", (r.tps_per_user, r.tps_per_gpu),
                    textcoords="offset points", xytext=(6, 4))
    ax.set_xlabel("TPS per user (user experience →)")
    ax.set_ylabel("TPS per GPU (cost efficiency →)")
    ax.set_title("Pareto: cost efficiency vs. user experience")
    plt.tight_layout()


### Notes

- Each sweep run writes to its own `-sweep-c<N>` artifact directory so exports never overwrite each other.
- You cannot optimize both axes at once: pick the point that meets your per-user SLO at the highest per-GPU efficiency, then cap each replica at that concurrency.
- A/B testing (vLLM vs. SGLang vs. TRT-LLM, or config changes): keep this exact command fixed and only change the endpoint — identical workload is what makes the numbers comparable.